<a href="https://colab.research.google.com/github/MuhammadAhmed196/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadAhmed196/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will use a Random Forest classifier for my Content Refresh Prioritization lane. The target is whether a page is likely to show a declining search-performance outcome, using only decision-time features. Random Forest fits this problem because refresh risk may depend on nonlinear interactions among visibility, clicks, position, content attributes, and other page-level signals. It also provides feature-importance summaries that help interpret which signals the model relies on. I will judge it by whether it improves on the Week-4 baseline on the same data, the same client-holdout split, and the same primary ranking metric rather than rewarding model complexity by itself.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a client-grouped holdout split. Entire clients, rather than individual pages, will be assigned to either the training set or the test set, so no client's pages appear in both sets. This is more honest for my content-refresh question because pages from the same client can share content, traffic, and measurement patterns. I will keep the split fixed with a random seed so the model and the Week-4 baseline are compared on the same held-out clients.

In [14]:
import os
import subprocess
import pandas as pd

# ---------------------------------------------------------
# Load the same starter data used for the Week-4 baseline
# ---------------------------------------------------------

REPO_DIR = "flyrank-ml-internship-starter"
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"

if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

data_path = os.path.join(
    REPO_DIR,
    "data",
    "raw",
    "content_refresh_anonymized.csv"
)

df = pd.read_csv(data_path)

# Observed target used for evaluation.
# This is the same target concept used by the Week-4 baseline.
df["is_declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

# ---------------------------------------------------------
# Features available before the decision
# ---------------------------------------------------------

feature_columns = [
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count"
]

target_column = "is_declining_label"
group_column = "client_id"

model_data = df.dropna(
    subset=[target_column]
).copy()

# ---------------------------------------------------------
# Client-grouped holdout
# ---------------------------------------------------------

from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_index, test_index = next(
    splitter.split(
        model_data,
        model_data[target_column],
        groups=model_data[group_column]
    )
)

train_data = model_data.iloc[train_index].copy()
test_data = model_data.iloc[test_index].copy()

train_clients = set(train_data[group_column])
test_clients = set(test_data[group_column])

overlap = train_clients.intersection(test_clients)

print("Training rows:", len(train_data))
print("Test rows:", len(test_data))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(overlap))

assert len(overlap) == 0

print("\nClient-holdout check: PASS")

Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Client overlap: 0

Client-holdout check: PASS


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    average_precision_score
)

# ---------------------------------------------------------
# RANDOM FOREST MODEL
# ---------------------------------------------------------

model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "random_forest",
        RandomForestClassifier(
            n_estimators=200,
            max_depth=6,
            min_samples_leaf=10,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    )
])

# Train only on the training clients
model.fit(
    train_data[feature_columns],
    train_data[target_column]
)

# Probability of the declining class
model_scores = model.predict_proba(
    test_data[feature_columns]
)[:, 1]

model_predictions = (
    model_scores >= 0.5
).astype(int)

# ---------------------------------------------------------
# WEEK-4 BASELINE ON THE SAME TEST SET
# ---------------------------------------------------------

baseline_scores = (
    (
        test_data["days_since_last_update"] >= 180
    ).astype(int)
    *
    (
        test_data["impressions_90d"] >= 500
    ).astype(int)
    *
    test_data["impressions_90d"]
)

# Top 50 rows by baseline score
baseline_order = np.argsort(
    -baseline_scores.to_numpy()
)

baseline_top50 = (
    test_data[target_column]
    .to_numpy()[baseline_order[:50]]
)

baseline_precision50 = baseline_top50.mean()

# Top 50 rows by model score
model_order = np.argsort(
    -model_scores
)

model_top50 = (
    test_data[target_column]
    .to_numpy()[model_order[:50]]
)

model_precision50 = model_top50.mean()

# Standard classification metrics
model_accuracy = accuracy_score(
    test_data[target_column],
    model_predictions
)

model_precision = precision_score(
    test_data[target_column],
    model_predictions,
    zero_division=0
)

model_ap = average_precision_score(
    test_data[target_column],
    model_scores
)

# ---------------------------------------------------------
# COMPARISON TABLE
# ---------------------------------------------------------

comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Random Forest"
    ],
    "Precision@50": [
        baseline_precision50,
        model_precision50
    ],
    "Accuracy": [
        np.nan,
        model_accuracy
    ],
    "Precision": [
        np.nan,
        model_precision
    ],
    "Average Precision": [
        np.nan,
        model_ap
    ]
})

comparison["Precision@50"] = comparison[
    "Precision@50"
].round(3)

comparison["Accuracy"] = comparison[
    "Accuracy"
].round(3)

comparison["Precision"] = comparison[
    "Precision"
].round(3)

comparison["Average Precision"] = comparison[
    "Average Precision"
].round(3)

print("MODEL vs BASELINE — SAME TEST CLIENTS")
display(comparison)

print(
    "\nBaseline Precision@50:",
    round(baseline_precision50, 3)
)

print(
    "Random Forest Precision@50:",
    round(model_precision50, 3)
)

print(
    "Random Forest Accuracy:",
    round(model_accuracy, 3)
)

print(
    "Random Forest Precision:",
    round(model_precision, 3)
)

print(
    "Random Forest Average Precision:",
    round(model_ap, 3)
)


MODEL vs BASELINE — SAME TEST CLIENTS


,method,Precision@50,Accuracy,Precision,Average Precision
0,Week-4 baseline,0.62,NaN,NaN,NaN
1,Random Forest,0.56,0.57,0.581,0.576



Baseline Precision@50: 0.62
Random Forest Precision@50: 0.56
Random Forest Accuracy: 0.57
Random Forest Precision: 0.581
Random Forest Average Precision: 0.576


The Random Forest did not beat the Week-4 baseline on the held-out clients. The baseline achieved Precision@50 of 0.620, while the Random Forest achieved 0.560. Because both methods were evaluated on the same held-out clients and the same ranking metric, the baseline remains the stronger method for this split. I will not treat the model as an improvement unless it can outperform the baseline under the same honest evaluation setup.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [11]:
# ---------------------------------------------------------
# ERROR ANALYSIS
# ---------------------------------------------------------

error_columns = [
    "client_id",
    "content_id",
    target_column,
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "search_volume"
]

error_analysis = test_data[error_columns].copy()

error_analysis["model_score"] = model_scores
error_analysis["model_prediction"] = model_predictions

# False positives:
# model predicted decline, but actual label is 0
false_positives = error_analysis[
    (error_analysis["model_prediction"] == 1) &
    (error_analysis[target_column] == 0)
].copy()

# False negatives:
# model predicted no decline, but actual label is 1
false_negatives = error_analysis[
    (error_analysis["model_prediction"] == 0) &
    (error_analysis[target_column] == 1)
].copy()

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nTop false positives by model score:")
display(
    false_positives
    .sort_values("model_score", ascending=False)
    .head(10)
)

print("\nTop false negatives by model score:")
display(
    false_negatives
    .sort_values("model_score", ascending=True)
    .head(10)
)

False positives: 1288
False negatives: 1362

Top false positives by model score:


,client_id,content_id,is_declining_label,impressions_90d,avg_position,ctr,content_age_days,days_since_last_update,word_count,search_volume,model_score,model_prediction
12472,client_8527a891e2,content_884c401ce126,0,101,23.1,0.00,174,92,1615.0,0.0,0.779811,1
12332,client_8527a891e2,content_4d9f36001f06,0,3369,13.2,0.03,275,104,1643.0,0.0,0.771234,1
4050,client_4e07408562,content_500bd3907331,0,4037,5.5,0.10,230,104,1294.0,20.0,0.771188,1
22042,client_8527a891e2,content_2ba626fea4d6,0,360,7.2,0.00,275,104,1405.0,10.0,0.769944,1
10080,client_8527a891e2,content_35d63627bf3e,0,1525,32.6,0.00,238,103,1592.0,20.0,0.769808,1
11061,client_8527a891e2,content_0b47dae0c7f9,0,1191,23.1,0.00,238,103,1514.0,20.0,0.769451,1
22461,client_4e07408562,content_7e3be2e230f5,0,909,33.4,0.11,280,104,1415.0,20.0,0.769305,1
22524,client_8527a891e2,content_846bb4dd8b44,0,870,17.6,0.11,275,104,1492.0,10.0,0.768707,1
5477,client_8527a891e2,content_3164f3076003,0,2696,16.1,0.04,275,104,1274.0,110.0,0.768471,1
20736,client_8527a891e2,content_41baf0722ad9,0,3115,12.8,0.00,275,104,1596.0,0.0,0.768314,1



Top false negatives by model score:


,client_id,content_id,is_declining_label,impressions_90d,avg_position,ctr,content_age_days,days_since_last_update,word_count,search_volume,model_score,model_prediction
27271,client_8527a891e2,content_7bc32bc1df59,1,1,0.0,0.0,238,92,1429.0,20.0,0.064391,0
1864,client_e629fa6598,content_16f38acf0f26,1,2,50.0,0.0,358,20,1590.0,20.0,0.153694,0
15676,client_8527a891e2,content_599f9c953f91,1,1,37.0,0.0,348,20,4041.0,50.0,0.168909,0
13114,client_8527a891e2,content_8f222654e93f,1,2,5.5,0.0,273,20,1994.0,0.0,0.170909,0
25350,client_8527a891e2,content_a4c38287770e,1,2,5.0,0.0,275,20,1495.0,0.0,0.173002,0
28072,client_8527a891e2,content_2847e276c475,1,1,6.0,0.0,271,20,1464.0,0.0,0.175816,0
18423,client_8527a891e2,content_77e2a54525b6,1,1,7.0,0.0,273,20,1449.0,10.0,0.177297,0
27395,client_8527a891e2,content_e72e6c56f0a3,1,1,9.0,0.0,273,20,1320.0,10.0,0.177605,0
14035,client_8527a891e2,content_025344275bba,1,2,39.0,0.0,348,20,3447.0,50.0,0.177702,0
4561,client_8527a891e2,content_b04d16f984c7,1,1,7.0,0.0,271,20,1948.0,0.0,0.178898,0


In [12]:
# ---------------------------------------------------------
# FEATURE IMPORTANCE
# ---------------------------------------------------------

rf_model = model.named_steps["random_forest"]

feature_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": rf_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

feature_importance["importance"] = (
    feature_importance["importance"].round(3)
)

print("Random Forest feature importance:")
display(feature_importance)

Random Forest feature importance:


,feature,importance
0,impressions_90d,0.341
1,content_age_days,0.228
2,avg_position,0.217
3,word_count,0.102
4,ctr,0.061
5,days_since_last_update,0.050


The model produces both false positives and false negatives, showing that the available page-level signals do not perfectly separate declining from non-declining pages. The false positives include pages with very different levels of impressions, positions, and content ages, so a high predicted decline score does not always correspond to an observed decline. The false negatives include pages that later declined even though the model assigned them relatively low decline scores, showing that the model can miss some declining pages.

Feature-importance results show that impressions_90d was the most important feature in the Random Forest (0.341), followed by content_age_days (0.228) and avg_position (0.217). Word_count contributed less (0.102), while ctr (0.061) and days_since_last_update (0.050) had the smallest relative importance among the six features. These values describe how the fitted Random Forest used the available features; they should not be interpreted as causal effects on traffic decline.

Overall, the Random Forest achieved Precision@50 of 0.560 on the held-out clients, compared with 0.620 for the Week-4 baseline. Therefore, on this evaluation split, the learned model did not improve the ranking quality of the baseline. The baseline remains the stronger method for now, while the model and its errors provide evidence about which signals may be useful for future iterations.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.